In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import root_mean_squared_log_error, r2_score


In [2]:
df_train = pd.read_csv('train.csv')
df_train.head()

,order_line_id,order_id,customer_id,product_id,store_id,order_ts,tz_hint,channel,payment_method,ship_mode,...,gift_wrap,promo_channel_code,return_window_days,warehouse_zone,picker_shift,banner_id,session_clicks,device_type,coupon_code,returned
0,Ld66386349be49a,O4a24edd12102,C8adf5053c8,Pec1f652dc7,ST9c4229,2023-04-18 08:37:00,NaN,online,card,standard,...,0,P0,60,F,morning,159,6.0,web,NaN,0
1,L446123aa27830a,O94ef7450ffbe,C7f5d2f4f43,Pd7315da2b9,STa28cb7,2023-11-19 00:01:00,NaN,online,card,standard,...,0,P5,14,B,morning,284,12.0,web,CP89059,0
2,Le69e6e4c3cc2ee,O90f4c2645ed7,Cd278ad788c,P06a8ad9292,STd7b40f,2024-04-23 03:01:00,UTC,online,card,standard,...,0,P3,30,D,morning,131,7.0,app,NaN,1
3,L119540f9e707da,O8c4e2b29eded,C6c001bebb1,P9161c2c7bb,ST9a2f2a,2023-01-14 13:12:00,NaN,online,NaN,standard,...,1,P4,14,A,morning,33,9.0,web,CP19010,0
4,Lec03bcc21f4677,Ofed2f726c816,Cdb5409f41f,P7174c4cfc6,STdff6bd,2024-04-11 10:21:00,UTC,online,card,standard,...,1,P1,30,B,afternoon,95,10.0,tablet,NaN,0


In [3]:
df_train.shape

(1400005, 27)

In [4]:
df_train.columns

Index(['order_line_id', 'order_id', 'customer_id', 'product_id', 'store_id',
       'order_ts', 'tz_hint', 'channel', 'payment_method', 'ship_mode',
       'unit_price', 'quantity', 'discount', 'freight_cost', 'order_value',
       'line_no', 'items_in_order', 'gift_wrap', 'promo_channel_code',
       'return_window_days', 'warehouse_zone', 'picker_shift', 'banner_id',
       'session_clicks', 'device_type', 'coupon_code', 'returned'],
      dtype='str')

In [5]:
df_train.dtypes

order_line_id             str
order_id                  str
customer_id               str
product_id                str
store_id                  str
order_ts                  str
tz_hint                   str
channel                   str
payment_method            str
ship_mode                 str
unit_price            float64
quantity                int64
discount              float64
freight_cost          float64
order_value           float64
line_no                 int64
items_in_order          int64
gift_wrap               int64
promo_channel_code        str
return_window_days      int64
warehouse_zone            str
picker_shift              str
banner_id               int64
session_clicks        float64
device_type               str
coupon_code               str
returned                int64
dtype: object

In [6]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1400005 entries, 0 to 1400004
Data columns (total 27 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   order_line_id       1400005 non-null  str    
 1   order_id            1400005 non-null  str    
 2   customer_id         1400005 non-null  str    
 3   product_id          1400005 non-null  str    
 4   store_id            1400005 non-null  str    
 5   order_ts            1400005 non-null  str    
 6   tz_hint             1119674 non-null  str    
 7   channel             1400005 non-null  str    
 8   payment_method      1357827 non-null  str    
 9   ship_mode           1343587 non-null  str    
 10  unit_price          1400005 non-null  float64
 11  quantity            1400005 non-null  int64  
 12  discount            1357101 non-null  float64
 13  freight_cost        1343328 non-null  float64
 14  order_value         1400005 non-null  float64
 15  line_no             140000

In [7]:
df_train.describe()

,unit_price,quantity,discount,freight_cost,order_value,line_no,items_in_order,gift_wrap,return_window_days,banner_id,session_clicks,returned
count,1.400005e+06,1.400005e+06,1.357101e+06,1.343328e+06,1.400005e+06,1.400005e+06,1.400005e+06,1.400005e+06,1.400005e+06,1.400005e+06,1.296295e+06,1.400005e+06
mean,5.810115e+01,4.953596e+01,2.675881e-01,1.367563e+01,6.141877e+02,2.121299e+01,4.142597e+01,5.002818e-01,2.923014e+01,1.603682e+02,9.002732e+00,1.805258e-01
std,8.811053e+02,1.318102e+02,2.718913e-01,1.365395e+01,4.396681e+03,5.420885e+01,9.158068e+01,5.000001e-01,1.628577e+01,9.188166e+01,2.994458e+00,3.846249e-01
min,6.000000e-02,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,7.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
25%,2.750000e+00,2.000000e+00,0.000000e+00,4.830000e+00,1.350000e+01,1.000000e+00,2.000000e+00,0.000000e+00,1.400000e+01,8.000000e+01,7.000000e+00,0.000000e+00
50%,6.300000e+00,5.000000e+00,1.600000e-01,9.180000e+00,3.400000e+01,3.000000e+00,5.000000e+00,1.000000e+00,3.000000e+01,1.600000e+02,9.000000e+00,0.000000e+00
75%,1.693000e+01,2.000000e+01,5.000000e-01,1.873000e+01,1.690000e+02,1.500000e+01,3.600000e+01,1.000000e+00,3.000000e+01,2.400000e+02,1.100000e+01,0.000000e+00
max,2.689410e+05,6.000000e+03,8.000000e-01,5.587000e+02,7.589210e+05,7.040000e+02,7.040000e+02,1.000000e+00,6.000000e+01,3.190000e+02,2.600000e+01,1.000000e+00


In [8]:
df_train.isnull().sum()

order_line_id               0
order_id                    0
customer_id                 0
product_id                  0
store_id                    0
order_ts                    0
tz_hint                280331
channel                     0
payment_method          42178
ship_mode               56418
unit_price                  0
quantity                    0
discount                42904
freight_cost            56677
order_value                 0
line_no                     0
items_in_order              0
gift_wrap                   0
promo_channel_code          0
return_window_days          0
warehouse_zone              0
picker_shift                0
banner_id                   0
session_clicks         103710
device_type                 0
coupon_code           1233664
returned                    0
dtype: int64

In [9]:
df_train.duplicated().sum()

np.int64(0)

In [10]:
df_customers = pd.read_csv('customers.csv')
df_customers.head()

,customer_id,region,customer_type,signup_date,age_band,newsletter_opt_in,preferred_contact,record_updated_at,loyalty_tier
0,Cd300d405d5,Brazil-Northeast,business,2023-10-18,45-54,0,email,2025-02-19,gold
1,C3f74c1f273,US-South,business,2022-12-26,25-34,1,none,2025-05-27,gold
2,C8ace5bf5cf,UK-SOUTH,small_business,2023-06-12,25-34,0,email,2024-06-01,gold
3,C86aa4a28f8,Brazil-Southeast,business,2022-09-19,25-34,1,none,2025-06-12,platinum
4,C51291b5c2a,Brazil-Southeast,business,2023-11-05,65+,0,email,2025-03-07,silver


In [11]:
df_customers.shape

(204930, 9)

In [12]:
df_customers.columns

Index(['customer_id', 'region', 'customer_type', 'signup_date', 'age_band',
       'newsletter_opt_in', 'preferred_contact', 'record_updated_at',
       'loyalty_tier'],
      dtype='str')

In [13]:
df_customers.dtypes

customer_id            str
region                 str
customer_type          str
signup_date            str
age_band               str
newsletter_opt_in    int64
preferred_contact      str
record_updated_at      str
loyalty_tier           str
dtype: object

In [14]:
df_customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 204930 entries, 0 to 204929
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   customer_id        204930 non-null  str  
 1   region             204930 non-null  str  
 2   customer_type      204930 non-null  str  
 3   signup_date        204930 non-null  str  
 4   age_band           188596 non-null  str  
 5   newsletter_opt_in  204930 non-null  int64
 6   preferred_contact  204930 non-null  str  
 7   record_updated_at  204930 non-null  str  
 8   loyalty_tier       204930 non-null  str  
dtypes: int64(1), str(8)
memory usage: 14.1 MB


In [15]:
df_customers.describe()

,newsletter_opt_in
count,204930.000000
mean,0.499341
std,0.500001
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


In [16]:
df_train.duplicated().sum()

np.int64(0)

In [17]:
df_products = pd.read_csv('products.csv')
df_products.head()

,product_id,category,list_price,weight_kg,length_cm,width_cm,height_cm,packaging_type,supplier_code,color,launch_date,review_count
0,Pcda2ea3ad3,health_beauty,41.27,0.30,20.0,16.0,16.0,standard,S3088,red,2021-08-08,1
1,Pe149c5bcae,sports_leisure,52.91,1.23,55.0,26.0,10.0,recycled,S2858,multi,2021-09-06,1
2,P7f490ab626,home_decor,93.77,0.30,45.0,35.0,15.0,recycled,S3025,black,2023-09-06,1
3,P2c3100e7ec,kitchen_dining,23.01,0.55,19.0,12.0,24.0,recycled,S3378,green,2022-03-30,1
4,P351bb6d542,fashion_accessories,80.21,0.25,22.0,15.0,11.0,standard,S2453,natural,2022-11-11,1


In [18]:
df_products.shape

(48677, 12)

In [19]:
df_products.columns

Index(['product_id', 'category', 'list_price', 'weight_kg', 'length_cm',
       'width_cm', 'height_cm', 'packaging_type', 'supplier_code', 'color',
       'launch_date', 'review_count'],
      dtype='str')

In [20]:
df_products.dtypes

product_id            str
category              str
list_price        float64
weight_kg         float64
length_cm         float64
width_cm          float64
height_cm         float64
packaging_type        str
supplier_code         str
color                 str
launch_date           str
review_count        int64
dtype: object

In [21]:
df_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 48677 entries, 0 to 48676
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      48677 non-null  str    
 1   category        48677 non-null  str    
 2   list_price      48677 non-null  float64
 3   weight_kg       46595 non-null  float64
 4   length_cm       48677 non-null  float64
 5   width_cm        48677 non-null  float64
 6   height_cm       48677 non-null  float64
 7   packaging_type  48677 non-null  str    
 8   supplier_code   48677 non-null  str    
 9   color           48677 non-null  str    
 10  launch_date     48677 non-null  str    
 11  review_count    48677 non-null  int64  
dtypes: float64(5), int64(1), str(6)
memory usage: 4.5 MB


In [22]:
df_products.describe()

,list_price,weight_kg,length_cm,width_cm,height_cm,review_count
count,48677.000000,46595.000000,48677.000000,48677.000000,48677.000000,48677.000000
mean,49.583911,2.038897,29.315056,22.138751,16.095713,1.028843
std,102.662349,4.133331,16.137515,11.708330,13.466592,0.491014
min,0.110000,0.000000,7.000000,6.000000,2.000000,0.000000
25%,10.050000,0.250000,18.000000,14.000000,7.000000,1.000000
50%,23.290000,0.570000,24.000000,18.000000,12.000000,1.000000
75%,52.480000,1.550000,35.000000,28.000000,20.000000,1.000000
max,7418.890000,40.420000,105.000000,118.000000,105.000000,57.000000


In [23]:
df_products.isnull().sum()

product_id           0
category             0
list_price           0
weight_kg         2082
length_cm            0
width_cm             0
height_cm            0
packaging_type       0
supplier_code        0
color                0
launch_date          0
review_count         0
dtype: int64

In [24]:
df_products.duplicated().sum()

np.int64(0)

In [25]:
df_calendar = pd.read_csv("calendar.csv")
df_calendar.head()

,week_start,holiday_flag,cpi,fuel_price,temperature_f,unemployment,fuel_surcharge
0,2022-12-26,1,167.73,2.718,34.0,8.62,0.272875
1,2023-01-02,0,167.83,2.694,34.2,8.62,0.279920
2,2023-01-09,0,167.87,2.672,37.7,8.62,0.251363
3,2023-01-16,0,167.91,2.684,39.2,8.62,0.260175
4,2023-01-23,0,167.95,2.731,42.9,8.62,0.221008


In [26]:
df_calendar.shape

(132, 7)

In [27]:
df_calendar.columns

Index(['week_start', 'holiday_flag', 'cpi', 'fuel_price', 'temperature_f',
       'unemployment', 'fuel_surcharge'],
      dtype='str')

In [28]:
df_calendar.dtypes

week_start            str
holiday_flag        int64
cpi               float64
fuel_price        float64
temperature_f     float64
unemployment      float64
fuel_surcharge    float64
dtype: object

In [29]:
df_calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   week_start      132 non-null    str    
 1   holiday_flag    132 non-null    int64  
 2   cpi             132 non-null    float64
 3   fuel_price      132 non-null    float64
 4   temperature_f   132 non-null    float64
 5   unemployment    132 non-null    float64
 6   fuel_surcharge  132 non-null    float64
dtypes: float64(5), int64(1), str(1)
memory usage: 7.3 KB


In [30]:
df_calendar.describe()

,holiday_flag,cpi,fuel_price,temperature_f,unemployment,fuel_surcharge
count,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000
mean,0.113636,171.580758,3.358129,60.604545,7.995985,0.330088
std,0.318578,3.103517,0.429420,15.334668,0.486199,0.070065
min,0.000000,167.550000,2.672000,30.500000,6.950000,0.169344
25%,0.000000,168.407500,2.887250,47.550000,7.510000,0.276244
50%,0.000000,171.395000,3.488000,61.000000,8.150000,0.324541
75%,0.000000,174.692500,3.725000,75.150000,8.430000,0.375593
max,1.000000,176.650000,3.998000,82.200000,8.620000,0.504282


In [31]:
df_calendar.isnull().sum()

week_start        0
holiday_flag      0
cpi               0
fuel_price        0
temperature_f     0
unemployment      0
fuel_surcharge    0
dtype: int64

In [32]:
df_calendar.duplicated().sum()

np.int64(0)

In [33]:
df_stores = pd.read_csv('stores.csv')
df_stores.head()

,store_id,channel,region,format,size_sqm,opened_date,manager_tenure_years,seller_rating
0,ST323bc0,marketplace,Brazil-Southeast,standard_seller,NaN,2018-03-03,3.8,2.066344
1,ST18d925,marketplace,Brazil-Southeast,standard_seller,NaN,2021-02-26,4.2,2.598503
2,ST624e50,marketplace,Brazil-Southeast,standard_seller,NaN,2012-08-12,2.9,2.189962
3,ST9a180e,marketplace,Brazil-South,standard_seller,NaN,2021-05-01,2.2,1.398380
4,ST921bef,marketplace,Brazil-Southeast,top_seller,NaN,2020-10-02,3.6,1.862539


In [34]:
df_stores.shape

(454, 8)

In [35]:
df_stores.columns

Index(['store_id', 'channel', 'region', 'format', 'size_sqm', 'opened_date',
       'manager_tenure_years', 'seller_rating'],
      dtype='str')

In [36]:
df_stores.dtypes

store_id                    str
channel                     str
region                      str
format                      str
size_sqm                float64
opened_date                 str
manager_tenure_years    float64
seller_rating           float64
dtype: object

In [37]:
df_stores.info()

<class 'pandas.DataFrame'>
RangeIndex: 454 entries, 0 to 453
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   store_id              454 non-null    str    
 1   channel               454 non-null    str    
 2   region                454 non-null    str    
 3   format                454 non-null    str    
 4   size_sqm              106 non-null    float64
 5   opened_date           454 non-null    str    
 6   manager_tenure_years  454 non-null    float64
 7   seller_rating         454 non-null    float64
dtypes: float64(3), str(5)
memory usage: 28.5 KB


In [38]:
df_stores.describe()

,size_sqm,manager_tenure_years,seller_rating
count,106.000000,454.000000,454.000000
mean,1803.367925,5.182819,2.511893
std,1881.537564,3.499393,0.843873
min,99.000000,0.000000,1.000000
25%,603.250000,2.500000,1.833104
50%,1256.500000,4.600000,2.442498
75%,1690.750000,7.100000,3.129979
max,8661.000000,24.200000,4.857608


In [39]:
df_stores.isnull().sum()

store_id                  0
channel                   0
region                    0
format                    0
size_sqm                348
opened_date               0
manager_tenure_years      0
seller_rating             0
dtype: int64

In [40]:
df_stores.duplicated().sum()

np.int64(0)

In [41]:
df_test = pd.read_csv("test.csv")
df_test.head()

,order_line_id,order_id,customer_id,product_id,store_id,order_ts,tz_hint,channel,payment_method,ship_mode,...,items_in_order,gift_wrap,promo_channel_code,return_window_days,warehouse_zone,picker_shift,banner_id,session_clicks,device_type,coupon_code
0,L85d608999fa1ee,O66cdc7b12362,Cc8b5f3d96b,P5078592efd,ST323bc0,2025-05-06 03:01:00,UTC,marketplace,card,express,...,3,0,P4,7,B,afternoon,14,6.0,web,NaN
1,L86272cb9543018,O69e96ddf8c3d,C2a37ede13c,P530b82380e,STa182be,2025-02-02 00:01:00,UTC,online,bank_slip,standard,...,4,0,P0,14,G,night,122,12.0,app,NaN
2,L66b5669bdace5b,O4681d81383a3,C4c68e20ad6,Pdc4270dad8,STa182be,2025-04-13 11:15:00,UTC,online,card,standard,...,28,0,P2,14,A,morning,82,2.0,app,NaN
3,L2c5886124612de,O582b7a1193da,Ce28248b56d,P8cd413bd5a,ST528fbe,2025-04-08 08:51:00,UTC,online,card,standard,...,24,1,P5,30,B,afternoon,254,11.0,web,NaN
4,L56fe7aa5ac5dd8,O6d3b847e3f34,Cb61e56229c,Pf5e42ac914,ST65524d,2025-01-05 13:28:00,NaN,online,NaN,standard,...,8,1,P0,14,B,morning,211,10.0,web,NaN


In [42]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 479995 entries, 0 to 479994
Data columns (total 26 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   order_line_id       479995 non-null  str    
 1   order_id            479995 non-null  str    
 2   customer_id         479995 non-null  str    
 3   product_id          479995 non-null  str    
 4   store_id            479995 non-null  str    
 5   order_ts            479995 non-null  str    
 6   tz_hint             384397 non-null  str    
 7   channel             479995 non-null  str    
 8   payment_method      465558 non-null  str    
 9   ship_mode           460734 non-null  str    
 10  unit_price          479995 non-null  float64
 11  quantity            479995 non-null  int64  
 12  discount            465349 non-null  float64
 13  freight_cost        460434 non-null  float64
 14  order_value         479995 non-null  float64
 15  line_no             479995 non-null  int64  


In [43]:
df_test.isnull().sum()

order_line_id              0
order_id                   0
customer_id                0
product_id                 0
store_id                   0
order_ts                   0
tz_hint                95598
channel                    0
payment_method         14437
ship_mode              19261
unit_price                 0
quantity                   0
discount               14646
freight_cost           19561
order_value                0
line_no                    0
items_in_order             0
gift_wrap                  0
promo_channel_code         0
return_window_days         0
warehouse_zone             0
picker_shift               0
banner_id                  0
session_clicks         35963
device_type                0
coupon_code           422327
dtype: int64

In [44]:
df_test.duplicated().sum()

np.int64(0)

In [45]:
# sns.pairplot(df_train.drop(columns = ['order_line_id', 'order_id', 'customer_id', 'product_id', 'store_id']))

In [46]:
# sns.pairplot(df_customers.drop(columns = 'customer_id'))

In [47]:
# sns.pairplot(df_products.drop(columns = 'id'))

In [48]:
# sns.pairplot(df_calendar.drop(columns = 'id'))

In [49]:
# sns.pairplot(df_stores.drop(columns = 'id'))